# Resonance Records — Perfilado de datos (Spotify + Grammy)

**Curso ETL (G01) — Workshop 2: Building a Reliable Batch Data Pipeline**
**Sección 6.3 — Data Profiling and Quality Risk Analysis**

Este notebook **solo recolecta evidencia**. No limpia, no transforma y no modifica ninguna fuente:
las decisiones de limpieza y las reglas de calidad se toman después, en `docs/quality_rules.md`.

| ID | Requerimiento analítico | Atributos que se perfilan |
|---|---|---|
| AR1 | Artistas con alto reconocimiento Grammy y baja popularidad en Spotify | Grammy: `artist`, `year`, `category` · Spotify: `artists`, `track_id`, `popularity` |
| AR2 | Familias de género donde se concentran los candidatos | Spotify: `track_genre`, `artists`, `popularity` · Grammy: `artist` |
| AR3 | Perfil sonoro de candidatos vs. consolidados | Spotify: `energy`, `valence`, `danceability`, `acousticness`, `explicit`, `popularity` |

**Alcance de columnas.** Solo estas columnas alimentan el pipeline; el resto de cada fuente se describe en
la tabla de estructura pero no se perfila en detalle:

| Fuente | Columnas en alcance |
|---|---|
| Grammy | `year`, `category`, `artist` |
| Spotify | `track_id`, `artists`, `popularity`, `explicit`, `track_genre`, `energy`, `valence`, `danceability`, `acousticness` |

**Fuentes:** Spotify se lee del CSV `data/raw/spotify_dataset.csv`; Grammy se lee de la **fuente operacional**
PostgreSQL (`grammy_source.public.grammy_awards`), no del CSV, igual que lo hará el pipeline.

**Salidas reproducibles:**
- `reports/*.html` — reportes automáticos de ydata-profiling.
- `docs/evidence/profiling/*.csv` y `*.png` — tablas y figuras que sustentan cada riesgo.
- `docs/evidence/profiling/profiling_risk_table.csv` — tabla Evidencia → Riesgo → Requerimiento.

**Requisitos para ejecutarlo:** el contenedor `data-db` encendido (`docker compose start`) y el kernel `.venv` seleccionado.

## 1. Configuración

Las rutas se calculan desde la raíz del proyecto (este notebook vive en `notebooks/`).
Las credenciales se leen del archivo `.env` local, así **ninguna contraseña queda escrita en el notebook**.
Desde Windows la base se alcanza por `localhost` y el puerto publicado `DATA_DB_HOST_PORT` (5433).

In [19]:
import os
import re
import sys
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

SPOTIFY_PATH = PROJECT_ROOT / "data" / "raw" / "spotify_dataset.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
EVIDENCE_DIR = PROJECT_ROOT / "docs" / "evidence" / "profiling"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# Chart style: one hue, recessive axes (single-series evidence figures)
SERIES_COLOR = "#2a78d6"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6,
    "axes.axisbelow": True, "font.size": 10,
})

M = {}  # metrics collected along the notebook; used to build the final risk table


def save_table(df, name, show=True):
    """Persist a profiling table as CSV evidence and display it."""
    path = EVIDENCE_DIR / f"{name}.csv"
    df.to_csv(path, index=True, encoding="utf-8")
    if show:
        display(df)
    print(f"-> saved {path.relative_to(PROJECT_ROOT)}")


def save_figure(fig, name):
    path = EVIDENCE_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"-> saved {path.relative_to(PROJECT_ROOT)}")


print("Project root:", PROJECT_ROOT)
print("Spotify CSV exists:", SPOTIFY_PATH.exists())

Project root: c:\dev\workshop-2
Spotify CSV exists: True


In [20]:
def load_env_file(path):
    """Minimal .env reader (KEY=VALUE lines). Values already set in the OS win."""
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip())


load_env_file(PROJECT_ROOT / ".env")
# From Windows the database is reached through the published port, not the Docker network
os.environ["DATA_DB_HOST"] = "localhost"
os.environ["DATA_DB_PORT"] = os.environ.get("DATA_DB_HOST_PORT", "5433")

from sqlalchemy import text  # noqa: E402
from src.db import source_engine  # noqa: E402

engine = source_engine()
with engine.connect() as conn:
    print("Connected to:", conn.execute(text("SELECT current_database()")).scalar(),
          "as", os.environ["DATA_DB_USER"])

Connected to: grammy_source as fer


## 2. Carga de las fuentes y verificación de estructura

Leer un archivo sin errores no garantiza que sirva para el análisis. Primero se verifica que existan
los atributos que exigen AR1–AR3. Ninguna columna se elimina ni se convierte aquí.

In [21]:
spotify = pd.read_csv(SPOTIFY_PATH)
if spotify.shape[1] == 1:
    raise ValueError("Spotify CSV parsed into a single column: check the delimiter.")

grammy = pd.read_sql(text("SELECT * FROM grammy_awards ORDER BY source_row_number"), engine)

print("Spotify:", spotify.shape)
print("Grammy :", grammy.shape)
display(spotify.head(3))
display(grammy.head(3))

Spotify: (114000, 21)
Grammy : (4810, 12)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,1,-6.746,0,0.1430,0.0322,0.000001,0.358,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.166,1,-17.235,1,0.0763,0.9240,0.000006,0.101,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.359,0,-9.734,1,0.0557,0.2100,0.000000,0.117,0.120,76.332,4,acoustic


,source_row_number,year,title,published_at,updated_at,category,nominee,artist,workers,img,winner,loaded_at
0,1,2019,62nd Annual GRAMMY Awards (2019),2020-05-19T05:10:28-07:00,2020-05-19T05:10:28-07:00,Record Of The Year,Bad Guy,Billie Eilish,"Finneas O'Connell, producer; Rob Kinelski & Fi...",https://www.grammy.com/sites/com/files/styles/...,True,2026-10-05 18:59:22.927586+00:00
1,2,2019,62nd Annual GRAMMY Awards (2019),2020-05-19T05:10:28-07:00,2020-05-19T05:10:28-07:00,Record Of The Year,"Hey, Ma",Bon Iver,"BJ Burton, Brad Cook, Chris Messina & Justin V...",https://www.grammy.com/sites/com/files/styles/...,True,2026-10-05 18:59:22.927586+00:00
2,3,2019,62nd Annual GRAMMY Awards (2019),2020-05-19T05:10:28-07:00,2020-05-19T05:10:28-07:00,Record Of The Year,7 rings,Ariana Grande,"Charles Anderson, Tommy Brown, Michael Foster ...",https://www.grammy.com/sites/com/files/styles/...,True,2026-10-05 18:59:22.927586+00:00


In [22]:
SPOTIFY_REQUIRED = {"track_id", "artists", "popularity", "explicit", "track_genre",
                    "energy", "valence", "danceability", "acousticness"}
GRAMMY_REQUIRED = {"year", "category", "artist"}

missing_spotify = SPOTIFY_REQUIRED - set(spotify.columns)
missing_grammy = GRAMMY_REQUIRED - set(grammy.columns)
print("Missing Spotify columns:", missing_spotify or "none")
print("Missing Grammy columns :", missing_grammy or "none")
assert not missing_spotify and not missing_grammy, "Required attributes are missing"


def structure(df, required):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "distinct": df.nunique(dropna=True),
        "in_scope": [c in required for c in df.columns],
    })


save_table(structure(spotify, SPOTIFY_REQUIRED), "01_spotify_structure")
save_table(structure(grammy, GRAMMY_REQUIRED), "01_grammy_structure")
M["spotify_rows"], M["grammy_rows"] = len(spotify), len(grammy)
M["spotify_index_col"] = next((c for c in spotify.columns if c.startswith("Unnamed")), None)

Missing Spotify columns: none
Missing Grammy columns : none


,dtype,non_null,distinct,in_scope
Unnamed: 0,int64,114000,114000,False
track_id,object,114000,89741,True
artists,object,113999,31437,True
album_name,object,113999,46589,False
track_name,object,113999,73608,False
popularity,int64,114000,101,True
duration_ms,int64,114000,50697,False
explicit,bool,114000,2,True
danceability,float64,114000,1174,True
energy,float64,114000,2083,True


-> saved docs\evidence\profiling\01_spotify_structure.csv


,dtype,non_null,distinct,in_scope
source_row_number,int64,4810,4810,False
year,int64,4810,62,True
title,object,4810,62,False
published_at,object,4810,4,False
updated_at,object,4810,10,False
category,object,4810,638,True
nominee,object,4804,4131,False
artist,object,2970,1658,True
workers,object,2620,2366,False
img,object,3443,1463,False


-> saved docs\evidence\profiling\01_grammy_structure.csv


> **Qué observar:** si aparece una columna `Unnamed: 0`, es un índice técnico que vino guardado en el CSV
> (no es un dato del negocio). No se borra aquí: se documenta y se decide en la transformación.

## 3. Reportes automáticos (ydata-profiling)

Se generan como artefactos HTML para revisión, igual que en la Activity-1.
Para Spotify (~114.000 filas) se usa `minimal=True`, que omite correlaciones e interacciones costosas
y evita agotar la memoria del equipo. El reporte **no decide** nada: es un apoyo para el perfilado dirigido de la sección 4.

In [23]:
from ydata_profiling import ProfileReport

reports = {
    "spotify_profiling.html": ProfileReport(spotify, title="Spotify — Data Profiling", minimal=True),
    "grammy_profiling.html": ProfileReport(grammy, title="Grammy (PostgreSQL source) — Data Profiling"),
}
for filename, report in reports.items():
    report.to_file(REPORTS_DIR / filename)
    print("-> saved", (REPORTS_DIR / filename).relative_to(PROJECT_ROOT))

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 21/21 [00:01<00:00, 19.90it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

...

100%|██████████| 12/12 [00:00<00:00, 95.09it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

-> saved reports\grammy_profiling.html


Abre los HTML de `reports/` en el navegador. Para verlos dentro del notebook puedes usar
`reports["grammy_profiling.html"].to_notebook_iframe()`, aunque consume bastante memoria.

## 4. Perfilado dirigido a los requerimientos

El enunciado exige *tablas o figuras que sustenten cada riesgo* (las capturas no bastan).
Cada subsección corresponde a una categoría de evidencia de la sección 6.3.

### 4.1 Completitud — valores faltantes

In [24]:
def completeness(df):
    nulls = df.isna().sum()
    blanks = df.select_dtypes(include="object").apply(lambda s: s.str.strip().eq("").sum())
    out = pd.DataFrame({"nulls": nulls, "blank_strings": blanks}).fillna(0).astype(int)
    out["pct_missing"] = ((out["nulls"] + out["blank_strings"]) / len(df) * 100).round(2)
    return out.sort_values("pct_missing", ascending=False)


save_table(completeness(spotify[sorted(SPOTIFY_REQUIRED)]), "02_spotify_completeness")
save_table(completeness(grammy[sorted(GRAMMY_REQUIRED)]), "02_grammy_completeness")

M["sp_null_artists"] = int(spotify["artists"].isna().sum())
M["gr_null_artist"] = int(grammy["artist"].isna().sum())
M["gr_null_artist_pct"] = round(M["gr_null_artist"] / len(grammy) * 100, 2)

,nulls,blank_strings,pct_missing
acousticness,0,0,0.0
artists,1,0,0.0
danceability,0,0,0.0
energy,0,0,0.0
explicit,0,0,0.0
popularity,0,0,0.0
track_genre,0,0,0.0
track_id,0,0,0.0
valence,0,0,0.0


-> saved docs\evidence\profiling\02_spotify_completeness.csv


,nulls,blank_strings,pct_missing
artist,1840,0,38.25
category,0,0,0.00
year,0,0,0.00


-> saved docs\evidence\profiling\02_grammy_completeness.csv


#### 4.1.1 ¿Dónde faltan los artistas de Grammy?

Si `artist` falta solo en ciertas categorías (por ejemplo, premios a productores o ingenieros), no es un
error aleatorio: es una característica de la fuente que afecta cuántos premios se pueden atribuir a un artista (AR1).

`nominee` y `workers` están **fuera de alcance**; solo se miran aquí para documentar si existía una alternativa
para recuperar el artista y justificar por qué no se usa (decisión que se registra en la transformación).

In [25]:
by_category = (
    grammy.assign(artist_missing=grammy["artist"].isna())
    .groupby("category")
    .agg(rows=("artist_missing", "size"), missing_artist=("artist_missing", "sum"))
)
by_category["pct_missing"] = (by_category["missing_artist"] / by_category["rows"] * 100).round(1)
by_category = by_category.sort_values("missing_artist", ascending=False)
save_table(by_category.head(20), "03_grammy_missing_artist_by_category")

missing = grammy[grammy["artist"].isna()]
alt = pd.DataFrame({
    "rows_missing_artist": [len(missing)],
    "with_nominee": [int(missing["nominee"].notna().sum())],
    "with_workers": [int(missing["workers"].notna().sum())],
    "categories_affected": [missing["category"].nunique()],
    "categories_always_missing": [int((by_category["pct_missing"] == 100).sum())],
})
save_table(alt, "03_grammy_missing_artist_alternatives")
M["gr_cat_always_missing"] = int(alt.loc[0, "categories_always_missing"])

display(missing[["year", "category", "nominee", "workers"]].sample(8, random_state=42))

,rows,missing_artist,pct_missing
category,,,
Song Of The Year,70,70,100.0
Best Opera Recording,64,64,100.0
Best Album Notes,63,63,100.0
Best Country Song,55,55,100.0
Best Instrumental Composition,55,55,100.0
Best New Artist,51,50,98.0
Best Historical Album,44,44,100.0
Best Chamber Music Performance,43,43,100.0
Best Instrumental Arrangement,39,39,100.0


-> saved docs\evidence\profiling\03_grammy_missing_artist_by_category.csv


,rows_missing_artist,with_nominee,with_workers,categories_affected,categories_always_missing
0,1840,1834,1654,251,246


-> saved docs\evidence\profiling\03_grammy_missing_artist_alternatives.csv


,year,category,nominee,workers
4234,1971,Best Album Cover,Pollution,"Gene Brownell, photographer; Dean O. Torrence,..."
3351,1987,Best Historical Album,Thelonious Monk - The Complete Riverside Recor...,(Thelonious Monk)
901,2013,Best American Roots Song,Love Has Come For You,"Edie Brickell & Steve Martin, songwriters (Ste..."
2963,1992,Best Instrumental Composition,Harlem Renaissance Suite,"Benny Carter, composer (Benny Carter)"
3923,1978,Best Album Package,Boys In The Trees,"Tony Lane & Johnny B. Lee, art directors (Carl..."
3921,1978,Best Arrangement Accompanying Vocal(s),Got To Get You Into My Life,"Maurice White, arranger (Earth, Wind & Fire)"
599,2017,Best Classical Instrumental Solo,Transcendental,"Daniil Trifonov; Sid McLauchlan, producer; Mar..."
2351,1999,Best Instrumental Arrangement,Chelsea Bridge,"Don Sebesky, arranger (Don Sebesky)"


In [26]:
top = by_category.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(top.index.str.slice(0, 55), top["missing_artist"], color=SERIES_COLOR, height=0.6)
ax.set_title("Grammy: registros sin artista por categoría (top 12)", loc="left", color="#0b0b0b")
ax.set_xlabel("Registros con artist nulo")
ax.grid(axis="y", visible=False)
save_figure(fig, "fig_grammy_missing_artist_by_category")
plt.show()

-> saved docs\evidence\profiling\fig_grammy_missing_artist_by_category.png


C:\Users\monte\AppData\Local\Temp\ipykernel_10380\845129778.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.2 Unicidad y duplicados

Grano esperado de Spotify: **una fila por canción** (`track_id`). Si un mismo `track_id` aparece varias veces,
hay que saber por qué (¿una fila por género?) y si la popularidad cambia entre copias, porque eso altera
los promedios de AR1–AR3.

In [27]:
dup_mask = spotify["track_id"].duplicated(keep=False)
dups = spotify[dup_mask]
per_track = dups.groupby("track_id").agg(
    copies=("track_id", "size"),
    genres=("track_genre", "nunique"),
    popularity_values=("popularity", "nunique"),
)

business_cols = sorted(SPOTIFY_REQUIRED)
uniq_spotify = pd.DataFrame({
    "metric": [
        "rows", "distinct track_id", "rows whose track_id repeats",
        "repeated track_ids", "repeated track_ids with >1 genre",
        "repeated track_ids with >1 popularity value",
        "exact duplicate rows (in-scope columns)",
    ],
    "value": [
        len(spotify), spotify["track_id"].nunique(), int(dup_mask.sum()),
        len(per_track), int((per_track["genres"] > 1).sum()),
        int((per_track["popularity_values"] > 1).sum()),
        int(spotify.duplicated(subset=business_cols).sum()),
    ],
}).set_index("metric")
save_table(uniq_spotify, "04_spotify_uniqueness")

M["sp_distinct_tracks"] = int(spotify["track_id"].nunique())
M["sp_dup_rows"] = int(dup_mask.sum())
M["sp_dup_tracks"] = len(per_track)
M["sp_dup_multi_genre"] = int((per_track["genres"] > 1).sum())
M["sp_dup_multi_pop"] = int((per_track["popularity_values"] > 1).sum())
M["sp_exact_dup_rows"] = int(spotify.duplicated(subset=business_cols).sum())

display(per_track.sort_values("copies", ascending=False).head(5))

,value
metric,
rows,114000
distinct track_id,89741
rows whose track_id repeats,40900
repeated track_ids,16641
repeated track_ids with >1 genre,16299
repeated track_ids with >1 popularity value,720
exact duplicate rows (in-scope columns),450


-> saved docs\evidence\profiling\04_spotify_uniqueness.csv


,copies,genres,popularity_values
track_id,,,
6S3JlDAGk3uu3NtZbPnuhS,9,9,1
2Ey6v4Sekh3Z0RUSISRosD,8,8,1
2kkvB3RNRzwjFdGhaUA0tz,8,8,1
5ZsAhuQ24mWHiduaxJqnhW,7,7,1
5ftfVzSLIi5ZxYdNbRtf41,7,7,1


In [28]:
grammy_key = ["year", "category", "artist"]
uniq_grammy = pd.DataFrame({
    "metric": ["rows", "distinct source_row_number",
               "duplicate rows on (year, category, artist), artist not null"],
    "value": [len(grammy), grammy["source_row_number"].nunique(),
              int(grammy.dropna(subset=["artist"]).duplicated(subset=grammy_key).sum())],
}).set_index("metric")
save_table(uniq_grammy, "04_grammy_uniqueness")
M["gr_dup_key"] = int(grammy.dropna(subset=["artist"]).duplicated(subset=grammy_key).sum())

# Repeated (year, category, artist) can be legitimate: one artist winning the same category twice in a year
display(grammy.dropna(subset=["artist"])
        .loc[lambda d: d.duplicated(subset=grammy_key, keep=False)]
        .sort_values(grammy_key)[grammy_key + ["nominee"]].head(10))

,value
metric,
rows,4810
distinct source_row_number,4810
"duplicate rows on (year, category, artist), artist not null",2


-> saved docs\evidence\profiling\04_grammy_uniqueness.csv


,year,category,artist,nominee
303,2019,Best Compilation Soundtrack For Visual Media,(Various Artists),The Lion King: The Songs
304,2019,Best Compilation Soundtrack For Visual Media,(Various Artists),Quentin Tarantino's Once Upon A Time In Hollywood
306,2019,Best Compilation Soundtrack For Visual Media,(Various Artists),Spider-Man: Into The Spider-Verse


### 4.3 Contenido categórico

- `track_genre` de Spotify tiene muchas etiquetas finas; AR2 necesita agruparlas en **familias de género**.
- `artists` de Spotify puede contener varios artistas separados por `;`.
- `artist` de Grammy puede traer colaboraciones en un solo texto (`Featuring`, `&`, `,`), lo que dificulta el cruce.

In [29]:
genres = spotify["track_genre"].value_counts()
M["sp_genres"] = int(genres.size)
save_table(genres.rename("tracks").to_frame(), "05_spotify_genres", show=False)
print("Distinct track_genre:", genres.size)
print("Tracks per genre — min / median / max:", genres.min(), genres.median(), genres.max())
print("\nSample of labels:", sorted(genres.index)[:40])

multi = spotify["artists"].str.contains(";", na=False)
M["sp_multi_artist_pct"] = round(multi.mean() * 100, 2)
print(f"\nSpotify rows with several artists (';'): {multi.sum()} ({M['sp_multi_artist_pct']} %)")

-> saved docs\evidence\profiling\05_spotify_genres.csv
Distinct track_genre: 114
Tracks per genre — min / median / max: 1000 1000.0 1000

Sample of labels: ['acoustic', 'afrobeat', 'alt-rock', 'alternative', 'ambient', 'anime', 'black-metal', 'bluegrass', 'blues', 'brazil', 'breakbeat', 'british', 'cantopop', 'chicago-house', 'children', 'chill', 'classical', 'club', 'comedy', 'country', 'dance', 'dancehall', 'death-metal', 'deep-house', 'detroit-techno', 'disco', 'disney', 'drum-and-bass', 'dub', 'dubstep', 'edm', 'electro', 'electronic', 'emo', 'folk', 'forro', 'french', 'funk', 'garage', 'german']

Spotify rows with several artists (';'): 30075 (26.38 %)


In [30]:
patterns = {
    "featuring / feat.": r"\b(?:featuring|feat\.?|ft\.)\s",
    "ampersand (&)": r"\s&\s",
    "comma (,)": r",",
    "' and '": r"\sand\s",
    "' with '": r"\swith\s",
    "Various Artists": r"various artists",
}
artist_text = grammy["artist"].dropna()
composite = pd.DataFrame({
    "rows": [int(artist_text.str.contains(p, case=False, regex=True).sum()) for p in patterns.values()]
}, index=list(patterns.keys()))
composite["pct_of_non_null_artist"] = (composite["rows"] / len(artist_text) * 100).round(2)
save_table(composite, "05_grammy_artist_composite_patterns")

any_composite = artist_text.str.contains("|".join(patterns.values()), case=False, regex=True)
M["gr_composite_pct"] = round(any_composite.mean() * 100, 2)

# Formatting variation: whitespace and case
M["gr_artist_untrimmed"] = int((artist_text != artist_text.str.strip()).sum())
M["gr_artist_case_variants"] = int(artist_text.str.strip().nunique() - artist_text.str.strip().str.lower().nunique())
print("Artist values with leading/trailing spaces:", M["gr_artist_untrimmed"])
print("Artist names differing only by upper/lower case:", M["gr_artist_case_variants"])

print("\nwinner values:", grammy["winner"].value_counts(dropna=False).to_dict())
M["gr_winner_values"] = grammy["winner"].value_counts(dropna=False).to_dict()
print("Distinct categories:", grammy["category"].nunique())
display(grammy["artist"].value_counts().head(15).rename("records"))

,rows,pct_of_non_null_artist
featuring / feat.,87,2.93
ampersand (&),411,13.84
"comma (,)",124,4.18
' and ',60,2.02
' with ',21,0.71
Various Artists,85,2.86


-> saved docs\evidence\profiling\05_grammy_artist_composite_patterns.csv
Artist values with leading/trailing spaces: 0
Artist names differing only by upper/lower case: 2

winner values: {'True': 4810}
Distinct categories: 638


artist
(Various Artists)    66
U2                   18
Aretha Franklin      16
Stevie Wonder        13
Ella Fitzgerald      13
Beyoncé              13
Bruce Springsteen    13
Jimmy Sturr          12
Dixie Chicks         12
Ray Charles          12
Tony Bennett         12
Adele                11
B.B. King            11
Eminem               11
Vince Gill           10
Name: records, dtype: int64

### 4.4 Contenido numérico

Rangos documentados del dataset de Spotify: `popularity` 0–100; `energy`, `valence`, `danceability` y
`acousticness` 0–1. Aquí solo se **mide** cuántos valores los violan.

In [31]:
ranges = {
    "popularity": (0, 100), "energy": (0, 1), "valence": (0, 1),
    "danceability": (0, 1), "acousticness": (0, 1),
}
rows = []
for col, (lo, hi) in ranges.items():
    s = pd.to_numeric(spotify[col], errors="coerce")
    out = (s < lo) | ((s > hi) if hi is not None else False)
    rows.append({"column": col, "min": s.min(), "p50": s.median(), "max": s.max(),
                 "nulls": int(s.isna().sum()), "out_of_range": int(out.sum()),
                 "documented_range": f"[{lo}, {hi if hi is not None else '∞'}]"})
numeric = pd.DataFrame(rows).set_index("column")
save_table(numeric, "06_spotify_numeric_ranges")
M["sp_pop_out_of_range"] = int(numeric.loc["popularity", "out_of_range"])
M["sp_audio_out_of_range"] = int(numeric.drop(index="popularity")["out_of_range"].sum())

pop0 = int((spotify["popularity"] == 0).sum())
M["sp_pop_zero"], M["sp_pop_zero_pct"] = pop0, round(pop0 / len(spotify) * 100, 2)
print(f"Tracks with popularity = 0: {pop0} ({M['sp_pop_zero_pct']} %)")
M["sp_explicit_values"] = spotify["explicit"].value_counts(dropna=False).to_dict()
print("explicit values:", M["sp_explicit_values"])

,min,p50,max,nulls,out_of_range,documented_range
column,,,,,,
popularity,0.0,35.000,100.000,0,0,"[0, 100]"
energy,0.0,0.685,1.000,0,0,"[0, 1]"
valence,0.0,0.464,0.995,0,0,"[0, 1]"
danceability,0.0,0.580,0.985,0,0,"[0, 1]"
acousticness,0.0,0.169,0.996,0,0,"[0, 1]"


-> saved docs\evidence\profiling\06_spotify_numeric_ranges.csv
Tracks with popularity = 0: 16020 (14.05 %)
explicit values: {False: 104253, True: 9747}


In [32]:
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.hist(spotify["popularity"], bins=50, color=SERIES_COLOR, edgecolor="#fcfcfb", linewidth=0.8)
ax.set_title("Spotify: distribución de popularity", loc="left", color="#0b0b0b")
ax.set_xlabel("popularity (0–100)")
ax.set_ylabel("Canciones")
ax.grid(axis="x", visible=False)
save_figure(fig, "fig_spotify_popularity_distribution")
plt.show()

-> saved docs\evidence\profiling\fig_spotify_popularity_distribution.png


C:\Users\monte\AppData\Local\Temp\ipykernel_10380\61987303.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.5 Contenido temporal

- Grammy: cobertura de años y si `published_at` / `updated_at` se pueden interpretar como fechas.
- Spotify no tiene fecha: `popularity` es una **foto** del momento en que se extrajo el dataset (limitación para AR1).

In [33]:
years = pd.to_numeric(grammy["year"], errors="coerce")
per_year = years.value_counts().sort_index()
expected = set(range(int(years.min()), int(years.max()) + 1))
missing_years = sorted(expected - set(per_year.index.astype(int)))

temporal = pd.DataFrame({
    "metric": ["min year", "max year", "distinct years", "missing years in range",
               "published_at unparseable", "updated_at unparseable"],
    "value": [int(years.min()), int(years.max()), int(per_year.size), str(missing_years or "none"),
              int(pd.to_datetime(grammy["published_at"], errors="coerce", utc=True).isna().sum()),
              int(pd.to_datetime(grammy["updated_at"], errors="coerce", utc=True).isna().sum())],
}).set_index("metric")
save_table(temporal, "07_grammy_temporal")
M["gr_year_min"], M["gr_year_max"] = int(years.min()), int(years.max())
M["gr_missing_years"] = missing_years

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(per_year.index, per_year.values, color=SERIES_COLOR, linewidth=2)
ax.set_title("Grammy: registros por año de ceremonia", loc="left", color="#0b0b0b")
ax.set_xlabel("year")
ax.set_ylabel("Registros")
save_figure(fig, "fig_grammy_records_per_year")
plt.show()

,value
metric,
min year,1958
max year,2019
distinct years,62
missing years in range,none
published_at unparseable,0
updated_at unparseable,0


-> saved docs\evidence\profiling\07_grammy_temporal.csv
-> saved docs\evidence\profiling\fig_grammy_records_per_year.png


C:\Users\monte\AppData\Local\Temp\ipykernel_10380\1185925117.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.6 Consistencia entre fuentes — ¿cuántos artistas de Grammy aparecen en Spotify?

Es la evidencia más importante: AR1, AR2 y AR3 dependen del cruce por nombre de artista.
Se mide la tasa de coincidencia con dos variantes de comparación, **sin modificar los datos**:

- **exacta:** el texto tal cual;
- **normalizada:** minúsculas, sin tildes y con espacios colapsados.

La diferencia entre ambas muestra cuánto aporta normalizar. Lo que siga sin coincidir después de normalizar
(colaboraciones, nombres distintos o artistas ausentes del catálogo) es un riesgo que la transformación debe tratar.

In [34]:
def normalize_name(value):
    """Comparison key only (profiling). Not applied to the source data."""
    if pd.isna(value):
        return pd.NA
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", value).strip().lower()


sp_artists = (
    spotify[["track_id", "artists"]].dropna(subset=["artists"])
    .assign(artist=lambda d: d["artists"].str.split(";"))
    .explode("artist")
)
sp_artists["artist"] = sp_artists["artist"].str.strip()
sp_exact = set(sp_artists["artist"])
sp_norm = set(sp_artists["artist"].map(normalize_name).dropna())

gr = grammy.dropna(subset=["artist"])[["artist"]].copy()
gr["norm"] = gr["artist"].map(normalize_name)
gr["match_exact"] = gr["artist"].isin(sp_exact)
gr["match_norm"] = gr["norm"].isin(sp_norm)
gr["composite"] = gr["artist"].str.contains("|".join(patterns.values()), case=False, regex=True)

gr_distinct = gr.drop_duplicates("artist")
cross = pd.DataFrame({
    "metric": [
        "distinct Spotify artists (after splitting ';')",
        "distinct Grammy artist values",
        "Grammy distinct artists matched — exact",
        "Grammy distinct artists matched — normalized",
        "Grammy records (with artist) matched — normalized",
        "match rate, simple names — normalized",
        "match rate, composite names — normalized",
    ],
    "value": [
        len(sp_exact), len(gr_distinct),
        f"{gr_distinct['match_exact'].sum()} ({gr_distinct['match_exact'].mean():.1%})",
        f"{gr_distinct['match_norm'].sum()} ({gr_distinct['match_norm'].mean():.1%})",
        f"{gr['match_norm'].sum()} ({gr['match_norm'].mean():.1%})",
        f"{gr_distinct.loc[~gr_distinct['composite'], 'match_norm'].mean():.1%}",
        f"{gr_distinct.loc[gr_distinct['composite'], 'match_norm'].mean():.1%}",
    ],
}).set_index("metric")
save_table(cross, "08_cross_source_artist_match")

M["x_match_exact"] = round(gr_distinct["match_exact"].mean() * 100, 1)
M["x_match_norm"] = round(gr_distinct["match_norm"].mean() * 100, 1)
M["x_match_simple"] = round(gr_distinct.loc[~gr_distinct["composite"], "match_norm"].mean() * 100, 1)
M["x_match_composite"] = round(gr_distinct.loc[gr_distinct["composite"], "match_norm"].mean() * 100, 1)
M["x_grammy_distinct"] = len(gr_distinct)

,value
metric,
distinct Spotify artists (after splitting ';'),29858
distinct Grammy artist values,1658
Grammy distinct artists matched — exact,517 (31.2%)
Grammy distinct artists matched — normalized,534 (32.2%)
Grammy records (with artist) matched — normalized,1310 (44.1%)
"match rate, simple names — normalized",44.1%
"match rate, composite names — normalized",3.5%


-> saved docs\evidence\profiling\08_cross_source_artist_match.csv


In [35]:
unmatched = (gr.loc[~gr["match_norm"], "artist"].value_counts()
             .head(25).rename("grammy_records").to_frame())
save_table(unmatched, "08_cross_source_top_unmatched")

,grammy_records
artist,
(Various Artists),66
U2,18
Jimmy Sturr,12
Dixie Chicks,12
Pat Metheny Group,10
CeCe Winans,9
Shirley Caesar,9
The Manhattan Transfer,8
Bill Cosby,8


-> saved docs\evidence\profiling\08_cross_source_top_unmatched.csv


> **Interpretación esperada:** algunos artistas no coinciden porque **no están** en el catálogo de Spotify
> (no es un error de calidad, es cobertura); otros no coinciden por **formato** (colaboraciones en un solo texto,
> variaciones de escritura). Solo el segundo grupo se puede mejorar con reglas de transformación.

## 5. Tabla Evidencia → Riesgo → Requerimiento

Se construye con las métricas calculadas arriba, así se regenera sola si cambian los datos.
Esta tabla va al README (sección de perfilado) y es la base de `docs/quality_rules.md`.

In [36]:
risk_rows = [
    ("Grammy / artist",
     f"{M['gr_null_artist']} nulos ({M['gr_null_artist_pct']} %); {M['gr_cat_always_missing']} categorías nunca tienen artista",
     "Premios no atribuibles a un artista: el conteo de reconocimientos queda subestimado", "AR1, AR2, AR3"),
    ("Grammy / artist",
     f"{M['gr_composite_pct']} % de los valores no nulos son colaboraciones (featuring, &, comas)",
     "El texto compuesto no coincide con un artista individual de Spotify", "AR1"),
    ("Grammy ↔ Spotify / artist",
     f"Coincidencia de artistas distintos: exacta {M['x_match_exact']} %, normalizada {M['x_match_norm']} % "
     f"(simples {M['x_match_simple']} %, compuestos {M['x_match_composite']} %)",
     "Sin normalización y separación de colaboraciones se pierden candidatos válidos", "AR1, AR2, AR3"),
    ("Spotify / track_id",
     f"{M['sp_dup_rows']} filas con track_id repetido ({M['sp_dup_tracks']} ids); "
     f"{M['sp_dup_multi_genre']} con más de un género; {M['sp_dup_multi_pop']} con popularidad distinta",
     "Doble conteo de canciones e inflación de promedios de popularidad y audio", "AR1, AR3"),
    ("Spotify / track_genre",
     f"{M['sp_genres']} etiquetas de género distintas",
     "Demasiado granular para decidir; requiere agrupar en familias", "AR2"),
    ("Spotify / artists",
     f"{M['sp_null_artists']} nulos; {M['sp_multi_artist_pct']} % de filas con varios artistas (';')",
     "Filas sin artista no se pueden cruzar; las colaboraciones deben separarse", "AR1, AR2, AR3"),
    ("Spotify / popularity",
     f"{M['sp_pop_zero']} canciones con popularidad 0 ({M['sp_pop_zero_pct']} %); fuera de [0, 100]: {M['sp_pop_out_of_range']}",
     "Popularidad 0 puede ser valor real o ausencia de dato; sesga el umbral de 'baja popularidad'", "AR1, AR3"),
    ("Spotify / audio features",
     f"energy, valence, danceability, acousticness fuera de [0, 1]: {M['sp_audio_out_of_range']}; explicit = {M['sp_explicit_values']}",
     "Un valor fuera de escala distorsionaría el perfil sonoro", "AR3"),
    ("Spotify / popularity (temporal)",
     "El dataset no trae fecha de extracción",
     "La popularidad es una foto en un momento; no permite ver tendencia", "AR1 (limitación)"),
    ("Grammy / year",
     f"Cobertura {M['gr_year_min']}–{M['gr_year_max']}; años faltantes en el rango: {M['gr_missing_years'] or 'ninguno'}",
     "Un año fuera de rango o no numérico alteraría el periodo de reconocimiento", "AR1"),
    ("Grammy / winner (fuera de alcance)",
     f"winner = {M['gr_winner_values']}",
     "Cada registro es un premio ganado: el conteo de registros equivale a premios; no hay nominados", "AR1 (definición del KPI)"),
    ("Grammy / (year, category, artist)",
     f"{M['gr_dup_key']} registros repetidos en esa combinación",
     "Un mismo reconocimiento podría contarse dos veces", "AR1"),
]
risk_table = pd.DataFrame(risk_rows, columns=["Dataset / Atributo", "Evidencia de perfilado",
                                              "Riesgo potencial de calidad", "Requerimiento"])
risk_table.index = [f"PR{i:02d}" for i in range(1, len(risk_table) + 1)]
save_table(risk_table, "profiling_risk_table")
md_lines = ["| ID | " + " | ".join(risk_table.columns) + " |", "|---" * (len(risk_table.columns) + 1) + "|"]
md_lines += ["| " + idx + " | " + " | ".join(str(v) for v in row) + " |" for idx, row in risk_table.iterrows()]
(EVIDENCE_DIR / "profiling_risk_table.md").write_text("\n".join(md_lines), encoding="utf-8")
print("-> saved docs/evidence/profiling/profiling_risk_table.md")

,Dataset / Atributo,Evidencia de perfilado,Riesgo potencial de calidad,Requerimiento
PR01,Grammy / artist,1840 nulos (38.25 %); 246 categorías nunca tie...,Premios no atribuibles a un artista: el conteo...,"AR1, AR2, AR3"
PR02,Grammy / artist,21.28 % de los valores no nulos son colaboraci...,El texto compuesto no coincide con un artista ...,AR1
PR03,Grammy ↔ Spotify / artist,Coincidencia de artistas distintos: exacta 31....,Sin normalización y separación de colaboracion...,"AR1, AR2, AR3"
PR04,Spotify / track_id,40900 filas con track_id repetido (16641 ids);...,Doble conteo de canciones e inflación de prome...,"AR1, AR3"
PR05,Spotify / track_genre,114 etiquetas de género distintas,Demasiado granular para decidir; requiere agru...,AR2
PR06,Spotify / artists,1 nulos; 26.38 % de filas con varios artistas ...,Filas sin artista no se pueden cruzar; las col...,"AR1, AR2, AR3"
PR07,Spotify / popularity,16020 canciones con popularidad 0 (14.05 %); f...,Popularidad 0 puede ser valor real o ausencia ...,"AR1, AR3"
PR08,Spotify / audio features,"energy, valence, danceability, acousticness fu...",Un valor fuera de escala distorsionaría el per...,AR3
PR09,Spotify / popularity (temporal),El dataset no trae fecha de extracción,La popularidad es una foto en un momento; no p...,AR1 (limitación)
PR10,Grammy / year,Cobertura 1958–2019; años faltantes en el rang...,Un año fuera de rango o no numérico alteraría ...,AR1


-> saved docs\evidence\profiling\profiling_risk_table.csv
-> saved docs/evidence/profiling/profiling_risk_table.md


## 6. Conclusión del perfilado

Revisa la tabla de la sección 5 contra los números reales y ajusta la redacción si un riesgo no se confirma
(por ejemplo, si `sp_audio_out_of_range` es 0, ese riesgo se reporta como **no observado** y la regla se justifica
como contrato de la fuente, no por evidencia).

**Siguiente paso:** convertir los riesgos confirmados en reglas de calidad (Rule ID, dimensión, métrica,
umbral y severidad) en `docs/quality_rules.md`.